# Whisper Fine-Tune — Malay STT
Voice model training for `percubaan.com`  
Model: `openai/whisper-small` | Language: `ms` (Malay)

In [ ]:
# Cell 1 — Check GPU
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
else:
    print('WARNING: No GPU detected. Go to Runtime > Change runtime type > T4 GPU')

In [ ]:
# Cell 2 — Install requirements
!pip install -q torch transformers datasets librosa soundfile jiwer pandas numpy accelerate evaluate scikit-learn
print('All dependencies installed.')

In [ ]:
# Cell 3 — Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Set this to your Google Drive folder that contains the voice-model/ directory
DRIVE_FOLDER = '/content/drive/MyDrive/voice-model'  # <-- update if needed

import os
if not os.path.isdir(DRIVE_FOLDER):
    raise FileNotFoundError(f'Drive folder not found: {DRIVE_FOLDER}\nUpload your voice-model/ folder to Google Drive first.')

print(f'Drive folder: {DRIVE_FOLDER}')
print('Contents:', os.listdir(DRIVE_FOLDER))

In [ ]:
# Cell 4 — Copy data from Drive to Colab local storage (faster I/O during training)
import shutil

COLAB_DIR = '/content/voice-model'
os.makedirs(COLAB_DIR, exist_ok=True)

# Copy src/ and data/ from Drive
for folder in ['src', 'data']:
    src  = os.path.join(DRIVE_FOLDER, folder)
    dest = os.path.join(COLAB_DIR, folder)
    if os.path.isdir(src):
        shutil.copytree(src, dest, dirs_exist_ok=True)
        print(f'Copied {folder}/ → {dest}')
    else:
        print(f'WARNING: {src} not found — skipping')

# Copy requirements.txt
req_src = os.path.join(DRIVE_FOLDER, 'requirements.txt')
if os.path.isfile(req_src):
    shutil.copy(req_src, COLAB_DIR)

print()
print('Colab dir contents:', os.listdir(COLAB_DIR))
audio_dir = os.path.join(COLAB_DIR, 'data', 'audio')
wav_count = len([f for f in os.listdir(audio_dir) if f.endswith('.wav')]) if os.path.isdir(audio_dir) else 0
print(f'Audio files found: {wav_count}')

In [ ]:
# Cell 5 — Prepare dataset (resample, normalise, split)
import sys
sys.path.insert(0, f'{COLAB_DIR}/src')

# Override paths to point to Colab local dirs
import config
config.AUDIO_DIR        = f'{COLAB_DIR}/data/audio'
config.TRANSCRIPT_PATH  = f'{COLAB_DIR}/data/transcripts.csv'
config.DATA_DIR         = f'{COLAB_DIR}/data/processed'
config.OUTPUT_DIR       = f'{COLAB_DIR}/models'

import importlib, prepare_data
importlib.reload(prepare_data)
prepare_data.main()

In [ ]:
# Cell 6 — Fine-tune Whisper
# Estimated time: ~30-60 min on T4 depending on dataset size
import train as train_module
importlib.reload(train_module)
train_module.main()

In [ ]:
# Cell 7 — Evaluate model, print WER
import evaluate as eval_module
importlib.reload(eval_module)
eval_module.main()

# Print results summary
import json
results_path = f'{COLAB_DIR}/results.json'
if os.path.isfile(results_path):
    with open(results_path) as f:
        r = json.load(f)
    print(f"\n=== Results Summary ===")
    print(f"Average WER  : {r['avg_wer']:.4f} ({r['avg_wer']*100:.1f}%)")
    print(f"Sample count : {r['sample_count']}")
    print(f"Timestamp    : {r['timestamp']}")

In [ ]:
# Cell 8 — Save trained model back to Google Drive
DRIVE_MODEL_DIR = os.path.join(DRIVE_FOLDER, 'models')
os.makedirs(DRIVE_MODEL_DIR, exist_ok=True)

local_model_dir = f'{COLAB_DIR}/models'
if os.path.isdir(local_model_dir) and os.listdir(local_model_dir):
    shutil.copytree(local_model_dir, DRIVE_MODEL_DIR, dirs_exist_ok=True)
    print(f'Model saved to Drive: {DRIVE_MODEL_DIR}')
    print('Files:', os.listdir(DRIVE_MODEL_DIR))
else:
    print('WARNING: No model files found to save.')

# Also save results.json
if os.path.isfile(results_path):
    shutil.copy(results_path, os.path.join(DRIVE_FOLDER, 'results.json'))
    print('results.json saved to Drive.')

In [ ]:
# Cell 9 — Zip and download model to your local machine
import zipfile
from google.colab import files

zip_path = '/content/whisper-malay-finetuned.zip'
local_model_dir = f'{COLAB_DIR}/models'

with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for root, _, filenames in os.walk(local_model_dir):
        for fname in filenames:
            full_path = os.path.join(root, fname)
            arc_name  = os.path.relpath(full_path, local_model_dir)
            zf.write(full_path, arc_name)

size_mb = os.path.getsize(zip_path) / 1e6
print(f'Zip created: {zip_path} ({size_mb:.1f} MB)')
print('Starting download…')
files.download(zip_path)